# Data Collection

Kunitz domain proteins from the PDB were retrieved, filtered and analyzed. Filtering was carried oout by sequence length, resolution, Pfam belonging and duplicated removal.

## Data Retrieval and Initial Processing

Sequence accession numbers was retrieve from an advance search in PDB filtering for proteins annotated as Kunitz domains by Pfam (PF00014), with a resolution lower a 3.5 to ensure high quality experiment and a sequence length between 40 and 80 residues long, to avoid fragmented sequences.

```Bash
Query Summary:
Attributes
            Identifier = "PF00014"
            AND
            Annotation Type = "Pfam"
      AND
      Data Collection Resolution < 3.5
      AND
      Polymer Entity Sequence Length < 80
      AND
      Polymer Entity Sequence Length > 40
```
The ids were downloaded and save in the file `Files\ids_pdb.csv`

Protein structure and other features were retrieved from the PDB via its GraphQL API. The obtain JSON was process to create a Pandas DataFrame containing relevant attributes for further analysis. The script filters for sequences between 40 and 80 residues and includes only those with Pfam annotations.

In [1]:
import pandas as pd
import requests      # Import requests for HTTP requests to the PDB API

In [4]:
 #Prepare PDB IDs for the GraphQL query
ids_str = ''  # Initialize an empty string to build the list of IDs for the query
ids = pd.Series(pd.read_csv('Files/ids_pdb.csv', header=1)['Entry ID'])  # Read IDs into a pandas Series
for i in range(ids.shape[0]):
    ids_str += '"' + ids[i] + '",'  # Append each ID in quotes, separated by commas
ids_str = ids_str[:-1]  # Remove the trailing comma

# Define the GraphQL query to fetch data for the specified PDB entries
query = '''
{
  entries(entry_ids: [%s])
  {
    rcsb_id
    rcsb_entry_container_identifiers {
      entry_id
    }
    rcsb_entry_info {
      diffrn_resolution_high {
        value
      }
    }
    polymer_entities {
      entity_poly {
        pdbx_seq_one_letter_code_can
        rcsb_sample_sequence_length
      }
      polymer_entity_instances {
        rcsb_polymer_entity_instance_container_identifiers {
          auth_asym_id
        }
      }
      rcsb_polymer_entity_annotation {
        type
      }
      rcsb_polymer_entity_container_identifiers {
        reference_sequence_identifiers {
          database_accession
        }
      }
    }
  }
} ''' %ids_str #%s will be replaced with the ids_str

# Send the GET request to the PDB GraphQL endpoint
p = requests.get('https://data.rcsb.org/graphql?query=%s' % requests.utils.requote_uri(query))

# Parse the JSON response
j = p.json()
pdb = j['data']['entries']  # Extract the list of entries

# Initialize an empty list to store dictionaries of extracted data
df_pdb = []

# Loop through each PDB entry
for i in range(ids.shape[0]):
    resolution = float(pdb[i]['rcsb_entry_info']['diffrn_resolution_high']['value'])  # Extract resolution as float
    for j_entity in pdb[i]['polymer_entities']:  # Loop through polymer entities in the entry
        len_seq = int(j_entity['entity_poly']['rcsb_sample_sequence_length'])  # Get sequence length
        if len_seq > 80 or len_seq < 40:  # Filter sequences outside 40-80 residues
            continue
        seq = j_entity['entity_poly']['pdbx_seq_one_letter_code_can']  # Extract sequence
        annot_type = j_entity['rcsb_polymer_entity_annotation'][0]['type']  # Extract annotation type
        chain = j_entity['polymer_entity_instances'][0]['rcsb_polymer_entity_instance_container_identifiers']['auth_asym_id']  # Extract chain ID

        # Append a list of attributes to the df list
        df_pdb.append([ids[i], seq, len_seq, resolution, annot_type, chain])

# Create a Pandas DataFrame from the dictionary
df_pdb = pd.DataFrame(df_pdb, columns=['PDB_ID', 'Sequence', 'Length', 'Resolution', 'Annotation_type', 'Chain'])
display(df_pdb.head())

,PDB_ID,Sequence,Length,Resolution,Annotation_type,Chain
0,1AAL,RPDFCLEPPYTGPCRLRIIRYFYNAKAGLCQTFVYGGCRAKRNNFK...,58,1.85,Pfam,F
1,1AAP,RPDFCLEPPYTGPCLARIIRYFYNAKAGLCQTFVYGGCRAKRNNFK...,58,1.35,Pfam,I
2,1B0C,VREVCSEQAETGPCRAMISRWYFDVTEGKCAPFFYGGCGGNRNNFD...,58,1.80,Pfam,B
3,1BHC,RPDFCLEPPYTGPCKARIIRYFYNAKAGLCQTFVYGGCRAKRNNFK...,58,1.22,Pfam,A
4,1BPI,RPDFCLEPPYTGPCKARIIRYFYNAKAGLCQTFVGGGCRAKRNNFK...,58,1.80,Pfam,A


In [6]:
print( "The obtained PDB entries were: " + str(df_pdb.shape[0]))

The obtained PDB entries were: 140


The DataFrame (`df_pdb`) containing key protein attributes: *PDB ID*, *sequence*, *length*, *resolution*, *annotation type*, and *chain identifier*. This DataFrame serves as the foundational dataset for subsequent filtering and analysis steps within the workflow.

## Removing Duplicate Entries and Filtering by Annotation Type

The duplicate entries were remove and only those associated with 'Pfam' annotations were retained. Duplicates can arise from various sources, such as multiple chains within the same PDB entry or redundant data, and their removal is crucial for maintaining data integrity. By specifically filtering for 'Pfam' annotations, we ensure that the subsequent analysis is based exclusively on relevant Kunitz domains, thereby enhancing the specificity and accuracy of our study.

In [7]:
# Identify PDB_IDs that appear more than once in the DataFrame
duplicated_pdb_entries = df_pdb[df_pdb.duplicated(subset=['PDB_ID'], keep=False)]

# Extract the unique PDB_IDs that are duplicated
duplicated_pdb_ids = duplicated_pdb_entries['PDB_ID'].unique()

print(f"Found {len(duplicated_pdb_ids)} PDB_IDs with duplicate entries:")
display(pd.Series(duplicated_pdb_ids))

Found 6 PDB_IDs with duplicate entries:


0    1D0D
1    1EAW
2    1ZJD
3    3LDJ
4    3LDM
5    5XX7
dtype: str

As observed, some entries were redundantly annotated by both Gene Ontology (GO) and Pfam. For consistency and relevance to Kunitz domains, only those entries specifically annotated by Pfam were retained.

In [8]:
# Filter the DataFrame to keep only entries with 'Pfam' annotation type.
df_pdb = df_pdb.query('Annotation_type == "Pfam"').reset_index(drop=True)
df_pdb

,PDB_ID,Sequence,Length,Resolution,Annotation_type,Chain
0,1AAL,RPDFCLEPPYTGPCRLRIIRYFYNAKAGLCQTFVYGGCRAKRNNFK...,58,1.85,Pfam,F
1,1AAP,RPDFCLEPPYTGPCLARIIRYFYNAKAGLCQTFVYGGCRAKRNNFK...,58,1.35,Pfam,I
2,1B0C,VREVCSEQAETGPCRAMISRWYFDVTEGKCAPFFYGGCGGNRNNFD...,58,1.80,Pfam,B
3,1BHC,RPDFCLEPPYTGPCKARIIRYFYNAKAGLCQTFVYGGCRAKRNNFK...,58,1.22,Pfam,A
4,1BPI,RPDFCLEPPYTGPCKARIIRYFYNAKAGLCQTFVGGGCRAKRNNFK...,58,1.80,Pfam,A
...,...,...,...,...,...,...
130,7QIR,RPDFCLEPPYTGPCKARIIRYFYNAKAGLCQTFVYGGCRAKRNNFK...,58,1.09,Pfam,A
131,7QIS,RPDFCLEPPYTGPCKARIIRYFYNAKAGLCQTFVYGGCRAKRNNFK...,58,2.63,Pfam,J
132,7QIT,RPDFCLEPPYTGPCKARIIRYFYNAKAGLCQTFVYGGCRAKRNNFK...,58,2.80,Pfam,E
133,8PTI,VREVCSEQAETGPCRAMISRWYFDVTEGKCAPFFYGGCGGNRNNFD...,58,1.50,Pfam,B


In [9]:
print( "The obtained PDB entries were: " + str(df_pdb.shape[0]))

The obtained PDB entries were: 135


In [11]:
# Save the dataframe as a TSV file in the Files folder
df_pdb.to_csv("Files/df_pdb.tsv", sep="\t", index=False)